# 🎯 Unidad 4 — REINFORCE desde Cero con PyTorch

**Curso: Aprendizaje por Refuerzo Profundo en Español**  
Adaptación pedagógica basada en el curso de HuggingFace 🤗  
Autor: Jose Miguel Lara Rangel · [aicraft.mx](https://aicraft.mx)

---

### El notebook más 'desde cero' del curso

En los notebooks anteriores usamos herramientas como Stable-Baselines3 o RL-Zoo.
Este notebook es diferente: **implementamos REINFORCE en PyTorch línea por línea**,
sin ninguna librería de RL.

Construiremos y probaremos el algoritmo en dos entornos:

| Entorno | Estados | Acciones | Red neuronal | Meta |
|---------|---------|----------|-------------|------|
| 🏋️ **CartPole-v1** | 4 | 2 | 2 capas (16 neuronas) | Recompensa media ≥ 350 |
| 🚁 **PixelCopter** | 7 | 2 | 3 capas (64 neuronas) | Recompensa media ≥ 5 |

Al terminar habrás:
- ✅ Implementado la clase `Policy` como una red neuronal en PyTorch
- ✅ Entendido el bug educativo de `act()` y por qué importa conceptualmente
- ✅ Implementado la función `reinforce()` con el truco del retorno hacia atrás
- ✅ Entrenado y evaluado los dos agentes
- ✅ Visualizado las curvas de aprendizaje de ambos

> 📎 **Slides de referencia:** Capítulos 4.1, 4.2 y 4.3 del curso.
> ⏱️ **Tiempo estimado:** ~1.5 horas (CartPole: ~5 min · PixelCopter: ~60 min con GPU)

---
## 1 · Configurar el entorno de trabajo

### 1.1 · Una nota importante: `gym` vs `gymnasium`

Los notebooks anteriores usaron `gymnasium` (la versión moderna de OpenAI Gym).
**Este notebook usa `gym`** (la versión clásica) por una razón específica:

- `PixelCopter` pertenece a la librería `gym_pygame`, que aún no fue portada a `gymnasium`
- La API de `gym` difiere en pequeños detalles: `.reset()` devuelve solo `state` (sin `info`),
  y `.step()` devuelve 4 valores (sin `truncated`)

Cuando veas `state = env.reset()` sin desempaquetar tupla, o `state, reward, done, _ = env.step(action)`,
es porque estamos usando `gym` clásico.

### 1.2 · Instalar dependencias

In [ ]:
%%capture
!apt install python-opengl ffmpeg xvfb
!pip install pyvirtualdisplay pyglet==1.5.1
!pip install -r https://raw.githubusercontent.com/huggingface/deep-rl-class/main/notebooks/unit4/requirements-unit4.txt

> ⚠️ Si Colab solicita reiniciar el entorno de ejecución después de la instalación, acepta y
> ejecuta desde la sección 1.3.

### 1.3 · Iniciar pantalla virtual

In [ ]:
from pyvirtualdisplay import Display
virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()
print("✅ Pantalla virtual iniciada")

### 1.4 · Importar librerías

A diferencia de los notebooks anteriores, aquí importamos directamente PyTorch:


In [ ]:
import numpy as np
from collections import deque   # estructura de cola para calcular el promedio móvil
import matplotlib.pyplot as plt
%matplotlib inline

# PyTorch — la librería de deep learning que usaremos
import torch
import torch.nn as nn                      # módulos de redes neuronales
import torch.nn.functional as F            # funciones de activación (ReLU, softmax)
import torch.optim as optim                # optimizadores (Adam, SGD)
from torch.distributions import Categorical # distribución para acciones discretas

# Gym clásico (no gymnasium) — necesario para PixelCopter
import gym
import gym_pygame

# Para grabar videos
import imageio

print("✅ Librerías importadas")

### 1.5 · Verificar GPU y definir el dispositivo

En PyTorch, los tensores y los modelos deben estar en el mismo **dispositivo**.
Si hay GPU disponible, la usamos para acelerar los cálculos.
La variable `device` la usaremos en todo el notebook con `.to(device)`.

In [ ]:
# Detectar si hay GPU disponible
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")
if str(device) == 'cuda:0':
    print("✅ GPU detectada — el entrenamiento será más rápido")
else:
    print("⚠️  Sin GPU — CartPole funcionará bien, PixelCopter puede tardar mucho")
    print("   Ve a Entorno de ejecución → Cambiar tipo → GPU T4")

---
## 2 · Parte 1 — CartPole-v1 🏋️

### 2.1 · Crear y explorar el entorno

CartPole es el entorno perfecto para depurar una implementación nueva de RL:
- Es simple y predecible (física determinista)
- Si el código tiene un bug, CartPole lo revela rápido
- Si funciona en CartPole, probablemente funciona en entornos más complejos

| Variable del estado | Descripción | Rango |
|---------------------|-------------|-------|
| `obs[0]` | Posición del carrito | ~−2.4 a +2.4 |
| `obs[1]` | Velocidad del carrito | Sin límite |
| `obs[2]` | Ángulo del palo | ~−0.21 a +0.21 rad |
| `obs[3]` | Velocidad angular del palo | Sin límite |

**Acciones:** `0` = empujar a la izquierda · `1` = empujar a la derecha

In [ ]:
env_id = "CartPole-v1"
env      = gym.make(env_id)          # entorno de entrenamiento
eval_env = gym.make(env_id)          # entorno de evaluación (separado)

s_size = env.observation_space.shape[0]   # 4 variables de estado
a_size = env.action_space.n               # 2 acciones

print(f"Espacio de estados: {s_size} variables")
print(f"Espacio de acciones: {a_size} acciones posibles")
print(f"Observación de ejemplo: {env.observation_space.sample()}")

### 2.2 · La clase Policy — la red neuronal de REINFORCE

La política π<sub>θ</sub> es una red neuronal que toma un **estado** como entrada
y produce una **distribución de probabilidades** sobre las acciones como salida.

**Arquitectura que vamos a construir:**
```
Estado (4 valores)
       ↓
   [fc1: Linear(4 → 16)]  + ReLU
       ↓
   [fc2: Linear(16 → 2)]
       ↓
   Softmax → probabilidades (suman 1.0)
       ↓
   P(acción=0), P(acción=1)
```

La clase también incluye el método `act(state)` que:
1. Convierte el estado a un tensor de PyTorch
2. Pasa el estado por la red para obtener las probabilidades
3. Crea una distribución `Categorical` sobre esas probabilidades
4. **Muestrea** una acción de la distribución
5. Devuelve la acción y su log-probabilidad (necesaria para calcular la pérdida)

---
### 🏋️ EJERCICIO (opcional) — implementar `__init__` y `forward`

Implementa las dos funciones marcadas. El método `act()` ya está dado.

**Pistas para `__init__`:**
- `nn.Linear(entrada, salida)` crea una capa totalmente conectada
- Necesitas `fc1` (s_size → h_size) y `fc2` (h_size → a_size)

**Pistas para `forward`:**
- `F.relu(self.fc1(x))` aplica la primera capa con activación ReLU
- El resultado pasa por `fc2`
- La salida final pasa por `F.softmax(x, dim=1)` para convertirla en probabilidades

Si prefieres no hacerlo como ejercicio, salta a la celda de Solución.

In [ ]:
# 🏋️ EJERCICIO — completa __init__ y forward
class Policy(nn.Module):
    def __init__(self, s_size, a_size, h_size):
        super(Policy, self).__init__()
        # Define las dos capas lineales aquí
        # self.fc1 = ...
        # self.fc2 = ...

    def forward(self, x):
        # Pasa x por fc1 con ReLU
        # Luego pasa por fc2
        # Devuelve softmax de la salida
        pass

    def act(self, state):
        state = torch.from_numpy(state).float().unsqueeze(0).to(device)
        probs = self.forward(state).cpu()
        m = Categorical(probs)
        action = np.argmax(m)        # ← ¿ves el problema aquí?
        return action.item(), m.log_prob(action)

---
### ✅ Solución parcial — `__init__` y `forward` correctos

(El método `act()` tiene un bug intencional que resolveremos en el siguiente paso)

In [ ]:
# ✅ SOLUCIÓN PARCIAL — __init__ y forward correctos
# (act() tiene un bug intencional — lo resolveremos a continuación)
class Policy(nn.Module):
    def __init__(self, s_size, a_size, h_size):
        super(Policy, self).__init__()
        self.fc1 = nn.Linear(s_size, h_size)   # capa oculta
        self.fc2 = nn.Linear(h_size, a_size)   # capa de salida

    def forward(self, x):
        x = F.relu(self.fc1(x))                # capa 1 + activación ReLU
        x = self.fc2(x)                        # capa 2 (sin activación)
        return F.softmax(x, dim=1)             # convertir a probabilidades

    def act(self, state):
        state = torch.from_numpy(state).float().unsqueeze(0).to(device)
        probs = self.forward(state).cpu()
        m = Categorical(probs)
        action = np.argmax(m)        # ← BUG INTENCIONAL aquí
        return action.item(), m.log_prob(action)

### 2.3 · El bug educativo de `act()` — encuéntralo antes de leer

El método `act()` tiene un error intencional. Antes de ver la explicación,
observa la línea marcada con `# ← BUG INTENCIONAL` y pregúntate:

- ¿Qué hace `np.argmax(m)`?
- ¿Es `m` un array de NumPy? ¿O es un objeto `Categorical` de PyTorch?
- ¿Cuál es la diferencia entre elegir la acción con mayor probabilidad y **muestrear** una acción?

Ejecuta la siguiente celda para ver el error en acción:

In [ ]:
# Provocar el error para ver el mensaje
debug_policy = Policy(s_size, a_size, 64).to(device)
try:
    state = env.reset()
    action, log_prob = debug_policy.act(state)
    print(f"Acción: {action}")
except Exception as e:
    print(f"❌ Error: {type(e).__name__}: {e}")

### ¿Por qué falla y por qué importa?

El error dice que `log_prob` recibe un valor que no es un Tensor.
Pero el problema conceptual es más profundo:

| | `np.argmax(m)` ❌ | `m.sample()` ✅ |
|--|-------------------|-----------------|
| Tipo de política | **Determinista** — siempre elige la misma acción | **Estocástica** — muestrea según las probabilidades |
| Exploración | Ninguna — siempre explota | Natural — acciones menos probables pueden ocurrir |
| Devuelve | Un índice (int) de NumPy | Un Tensor de PyTorch |
| Compatible con `log_prob` | ❌ No | ✅ Sí |

En REINFORCE **necesitamos** una política estocástica. Si siempre elegimos la acción
más probable, nunca exploramos y el gradiente no tiene señal útil.

Recuerda: una de las ventajas de los métodos de política que vimos en las slides
es precisamente que la exploración es **intrínseca** — no necesitamos ε-greedy.
Pero eso solo funciona si realmente **muestreamos** la acción.

---
### ✅ Solución correcta — `act()` con `m.sample()`

In [ ]:
# ✅ SOLUCIÓN CORRECTA — Policy completa y funcional
class Policy(nn.Module):
    def __init__(self, s_size, a_size, h_size):
        super(Policy, self).__init__()
        self.fc1 = nn.Linear(s_size, h_size)
        self.fc2 = nn.Linear(h_size, a_size)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return F.softmax(x, dim=1)

    def act(self, state):
        state = torch.from_numpy(state).float().unsqueeze(0).to(device)
        probs = self.forward(state).cpu()
        m = Categorical(probs)
        action = m.sample()          # ✅ muestrear — no argmax
        return action.item(), m.log_prob(action)

# Verificar que ya no hay error
test_policy = Policy(s_size, a_size, 64).to(device)
state = env.reset()
action, log_prob = test_policy.act(state)
print(f"✅ Policy funciona correctamente")
print(f"   Acción: {action} | log_prob: {log_prob.item():.4f}")

### 2.4 · La función `reinforce()` — el corazón del algoritmo

Antes del ejercicio, entendamos las 4 partes de esta función:

**Parte 1: El promedio móvil**
```python
scores_deque = deque(maxlen=100)
```
`deque` con `maxlen=100` guarda solo los últimos 100 valores.
Cuando llega el elemento 101, el primero se descarta automáticamente.
Así `np.mean(scores_deque)` siempre da el promedio de los últimos 100 episodios.

**Parte 2: El bucle de episodio**
En cada episodio guardamos dos listas: `saved_log_probs` (log π(a|s) para cada paso)
y `rewards` (recompensa de cada paso). Las necesitamos al final para calcular la pérdida.

**Parte 3: El truco del retorno hacia atrás** ← el más importante

Necesitamos G_t para cada paso t. Podríamos calcularlo con un bucle hacia adelante,
pero sería O(N²). Hay un truco O(N) usando la propiedad recursiva de Bellman:

```
G_T   = r_T
G_T-1 = r_T-1 + γ·G_T
G_T-2 = r_T-2 + γ·G_T-1
...
G_0   = r_0   + γ·G_1
```

Ejemplo concreto con [r=1, r=1, r=1, r=1] y γ=1.0:
```
t=3: G_3 = 1
t=2: G_2 = 1 + 1.0×1 = 2
t=1: G_1 = 1 + 1.0×2 = 3
t=0: G_0 = 1 + 1.0×3 = 4
→ returns = [4, 3, 2, 1]  (G más alto para las acciones tempranas)
```

`appendleft()` añade al inicio de la cola en O(1), preservando el orden cronológico.

**Parte 4: La función de pérdida negativa**

PyTorch minimiza pérdidas. Para **maximizar** J(θ) = E[G·log π(a|s)],
definimos la pérdida como su negativo:
```python
policy_loss = [-log_prob * G for log_prob, G in zip(saved_log_probs, returns)]
```
Minimizar `−log π(a|s)·G` es equivalente a maximizar `log π(a|s)·G`.

---
### 🏋️ EJERCICIO (opcional) — completa los 3 huecos

Los tres huecos marcados con `### tu código aquí`:
1. `state = ...` — reiniciar el entorno
2. `action, log_prob = ...` — elegir acción con la política
3. `returns.appendleft(...)` — la ecuación del retorno hacia atrás: `γ·G_{t+1} + r_t`

Si prefieres no hacerlo como ejercicio, salta a la celda de Solución.

In [ ]:
# 🏋️ EJERCICIO — completa los 3 huecos marcados
def reinforce(policy, optimizer, n_training_episodes, max_t, gamma, print_every):
    scores_deque = deque(maxlen=100)
    scores = []

    for i_episode in range(1, n_training_episodes + 1):
        saved_log_probs = []
        rewards = []

        # HUECO 1: reiniciar el entorno
        state = ### tu código aquí

        for t in range(max_t):
            # HUECO 2: elegir acción con la política
            action, log_prob = ### tu código aquí
            saved_log_probs.append(log_prob)

            state, reward, done, _ = env.step(action)
            rewards.append(reward)
            if done:
                break

        scores_deque.append(sum(rewards))
        scores.append(sum(rewards))

        # Calcular retornos hacia atrás (truco O(N))
        returns = deque(maxlen=max_t)
        n_steps = len(rewards)
        for t in range(n_steps)[::-1]:          # de t=T hasta t=0
            disc_return_t = returns[0] if len(returns) > 0 else 0
            # HUECO 3: calcular G_t = r_t + gamma * G_{t+1}
            returns.appendleft(### tu código aquí)

        # Estandarizar los retornos para mayor estabilidad
        eps = np.finfo(np.float32).eps.item()
        returns = torch.tensor(returns)
        returns = (returns - returns.mean()) / (returns.std() + eps)

        # Calcular la pérdida (negativa para convertir ascenso en descenso)
        policy_loss = []
        for log_prob, disc_return in zip(saved_log_probs, returns):
            policy_loss.append(-log_prob * disc_return)
        policy_loss = torch.cat(policy_loss).sum()

        # Actualizar los pesos de la red
        optimizer.zero_grad()
        policy_loss.backward()
        optimizer.step()

        if i_episode % print_every == 0:
            print(f"Episodio {i_episode}\tPuntuación media (últimos 100): {np.mean(scores_deque):.2f}")

    return scores

---
### ✅ Solución — función `reinforce()` completa

In [ ]:
# ✅ SOLUCIÓN — función reinforce() completa y comentada
def reinforce(policy, optimizer, n_training_episodes, max_t, gamma, print_every):
    scores_deque = deque(maxlen=100)   # promedio móvil de los últimos 100 episodios
    scores = []                         # historial completo para graficar

    for i_episode in range(1, n_training_episodes + 1):
        saved_log_probs = []            # log π(a_t|s_t) de cada paso
        rewards = []                    # r_t de cada paso

        # ✅ HUECO 1: reiniciar el entorno
        state = env.reset()

        for t in range(max_t):
            # ✅ HUECO 2: la política elige la acción
            action, log_prob = policy.act(state)
            saved_log_probs.append(log_prob)

            state, reward, done, _ = env.step(action)
            rewards.append(reward)
            if done:
                break

        scores_deque.append(sum(rewards))
        scores.append(sum(rewards))

        # Calcular retornos hacia atrás en O(N)
        returns = deque(maxlen=max_t)
        n_steps = len(rewards)
        for t in range(n_steps)[::-1]:
            disc_return_t = returns[0] if len(returns) > 0 else 0
            # ✅ HUECO 3: G_t = r_t + gamma * G_{t+1}
            returns.appendleft(gamma * disc_return_t + rewards[t])

        # Estandarizar retornos — reduce varianza y estabiliza el entrenamiento
        eps = np.finfo(np.float32).eps.item()
        returns = torch.tensor(returns)
        returns = (returns - returns.mean()) / (returns.std() + eps)

        # Pérdida = −log π(a|s) × G  (negativa para que PyTorch pueda minimizarla)
        policy_loss = []
        for log_prob, disc_return in zip(saved_log_probs, returns):
            policy_loss.append(-log_prob * disc_return)
        policy_loss = torch.cat(policy_loss).sum()

        # Paso de descenso de gradiente
        optimizer.zero_grad()
        policy_loss.backward()
        optimizer.step()

        if i_episode % print_every == 0:
            print(f"Episodio {i_episode}\tPuntuación media (últimos 100): {np.mean(scores_deque):.2f}")

    return scores

### 2.5 · Hiperparámetros y entrenamiento de CartPole

| Parámetro | Valor | Justificación |
|-----------|-------|--------------|
| `h_size` | 16 | Red pequeña — CartPole es simple |
| `n_training_episodes` | 1000 | Suficiente para converger |
| `gamma` | 1.0 | Sin descuento — CartPole es corto y cada paso importa igual |
| `lr` | 1e-2 | Tasa alta — la pérdida es poco ruidosa en CartPole |

> ⏱️ El entrenamiento de CartPole tarda **~1-2 minutos** con o sin GPU.

In [ ]:
cartpole_hyperparameters = {
    "h_size": 16,
    "n_training_episodes": 1000,
    "n_evaluation_episodes": 10,
    "max_t": 1000,
    "gamma": 1.0,
    "lr": 1e-2,
    "env_id": env_id,
    "state_space": s_size,
    "action_space": a_size,
}

# Crear la política y el optimizador Adam
cartpole_policy = Policy(
    cartpole_hyperparameters["state_space"],
    cartpole_hyperparameters["action_space"],
    cartpole_hyperparameters["h_size"]
).to(device)

cartpole_optimizer = optim.Adam(
    cartpole_policy.parameters(),
    lr=cartpole_hyperparameters["lr"]
)

print("✅ Policy y optimizador creados")
print(f"Parámetros entrenables: {sum(p.numel() for p in cartpole_policy.parameters()):,}")

In [ ]:
print("Entrenando CartPole-v1...")
scores_cartpole = reinforce(
    cartpole_policy,
    cartpole_optimizer,
    cartpole_hyperparameters["n_training_episodes"],
    cartpole_hyperparameters["max_t"],
    cartpole_hyperparameters["gamma"],
    print_every=100
)
print("\n✅ Entrenamiento CartPole completado")

### 2.6 · Curva de aprendizaje de CartPole

Graficamos la puntuación media de los últimos 100 episodios a lo largo del entrenamiento.
Una buena implementación de REINFORCE debe mostrar una curva que sube consistentemente.

In [ ]:
plt.figure(figsize=(10, 4))
# Calcular media móvil de 100 episodios
rolling_mean = [np.mean(scores_cartpole[max(0, i-100):i+1])
                for i in range(len(scores_cartpole))]

plt.plot(scores_cartpole, alpha=0.3, color='#3b82f6', label='Puntuación por episodio')
plt.plot(rolling_mean, color='#22d3ee', linewidth=2, label='Media móvil (100 ep)')
plt.axhline(y=350, color='#22c55e', linestyle='--', alpha=0.7, label='Meta (≥350)')
plt.xlabel("Episodio")
plt.ylabel("Puntuación total")
plt.title("Curva de aprendizaje — CartPole-v1 (REINFORCE)")
plt.legend()
plt.tight_layout()
plt.savefig("curva_cartpole.png", dpi=100)
plt.show()
print(f"Puntuación final (media últimos 100 ep): {np.mean(scores_cartpole[-100:]):.2f}")

### 2.7 · Evaluar el agente de CartPole

La función `evaluate_agent` corre `n_eval_episodes` episodios completos usando
la política **sin exploración** (determinista) y calcula la recompensa media.

In [ ]:
def evaluate_agent(env, max_steps, n_eval_episodes, policy):
    """
    Evalúa la política durante n_eval_episodes episodios.
    Devuelve la recompensa media y su desviación estándar.
    """
    episode_rewards = []
    for episode in range(n_eval_episodes):
        state = env.reset()
        total_rewards_ep = 0
        done = False
        for step in range(max_steps):
            action, _ = policy.act(state)
            new_state, reward, done, info = env.step(action)
            total_rewards_ep += reward
            if done:
                break
            state = new_state
        episode_rewards.append(total_rewards_ep)
    mean_reward = np.mean(episode_rewards)
    std_reward  = np.std(episode_rewards)
    return mean_reward, std_reward

In [ ]:
mean_reward_cp, std_reward_cp = evaluate_agent(
    eval_env,
    cartpole_hyperparameters["max_t"],
    cartpole_hyperparameters["n_evaluation_episodes"],
    cartpole_policy
)
print(f"CartPole-v1 — Recompensa media: {mean_reward_cp:.2f} ± {std_reward_cp:.2f}")

if mean_reward_cp >= 350:
    print("✅ ¡Meta alcanzada! (≥350)")
elif mean_reward_cp >= 200:
    print("🟡 Buen resultado. Prueba con más episodios de entrenamiento.")
else:
    print("🔴 El agente necesita más entrenamiento.")

### 2.8 · Ver al agente de CartPole en video

In [ ]:
def grabar_video_policy(env_id, policy, archivo='replay.mp4', fps=30, **kwargs):
    """Graba un episodio completo del agente."""
    env_v = gym.make(env_id, **kwargs)
    frames = []
    state = env_v.reset()
    done = False
    while not done:
        frames.append(env_v.render(mode='rgb_array'))
        action, _ = policy.act(state)
        state, _, done, _ = env_v.step(action)
    env_v.close()
    imageio.mimsave(archivo, [np.array(f) for f in frames], fps=fps)
    return archivo

print("Grabando video de CartPole...")
video_cp = grabar_video_policy('CartPole-v1', cartpole_policy, 'replay_cartpole.mp4')

from IPython.display import Video
Video(video_cp, embed=True, width=400)

---
## 3 · Parte 2 — PixelCopter 🚁 (el mismo algoritmo, más difícil)

PixelCopter es un juego de vuelo en túnel. El helicóptero debe mantenerse en el
centro del corredor mientras aparecen obstáculos.

### 3.1 · Explorar el entorno

In [ ]:
env_id = "Pixelcopter-PLE-v0"
env      = gym.make(env_id)
eval_env = gym.make(env_id)

s_size = env.observation_space.shape[0]
a_size = env.action_space.n
print(f"Estados: {s_size} | Acciones: {a_size}")

**Las 7 variables del estado de PixelCopter:**

| Índice | Variable | Descripción |
|--------|----------|-------------|
| `obs[0]` | Posición y del helicóptero | Altura actual |
| `obs[1]` | Velocidad y | Velocidad vertical (positivo = sube) |
| `obs[2]` | Distancia al suelo del corredor | Qué tan lejos está el suelo del túnel |
| `obs[3]` | Distancia al techo del corredor | Qué tan lejos está el techo del túnel |
| `obs[4]` | Posición x del siguiente obstáculo | Distancia horizontal al próximo bloque |
| `obs[5]` | Posición y superior del obstáculo | Borde superior del bloque |
| `obs[6]` | Posición y inferior del obstáculo | Borde inferior del bloque |

Con 7 variables el agente puede inferir la trayectoria necesaria para esquivar obstáculos.

In [ ]:
print(f"Observación de ejemplo: {env.observation_space.sample()}")
print(f"Acciones: {a_size} (0=motor apagado/bajar, 1=motor encendido/subir)")

### 3.2 · Red neuronal más profunda para PixelCopter

PixelCopter es más complejo que CartPole: tiene más variables de estado y
la dinámica de vuelo es más difícil de aprender. Necesitamos una red más grande:

```
Estado (7 valores)
       ↓
   [fc1: Linear(7 → 64)]   + ReLU
       ↓
   [fc2: Linear(64 → 128)]  + ReLU    ← capa extra vs CartPole
       ↓
   [fc3: Linear(128 → 2)]
       ↓
   Softmax → P(bajar), P(subir)
```

---
### 🏋️ EJERCICIO (opcional) — implementar `__init__` y `forward` para PixelCopter

La arquitectura tiene **3 capas** (vs 2 de CartPole).
El método `act()` es idéntico al de CartPole — ya está dado.

**Pistas:**
- `fc1`: s_size → h_size
- `fc2`: h_size → h_size * 2   (capa intermedia más grande)
- `fc3`: h_size * 2 → a_size
- En `forward`: ReLU después de fc1 y fc2, softmax al final

In [ ]:
# 🏋️ EJERCICIO — Policy de 3 capas para PixelCopter
class Policy(nn.Module):
    def __init__(self, s_size, a_size, h_size):
        super(Policy, self).__init__()
        # Define las tres capas
        # self.fc1 = ...
        # self.fc2 = ...
        # self.fc3 = ...

    def forward(self, x):
        # ReLU después de fc1 y fc2
        # Softmax al final
        pass

    def act(self, state):
        state = torch.from_numpy(state).float().unsqueeze(0).to(device)
        probs = self.forward(state).cpu()
        m = Categorical(probs)
        action = m.sample()
        return action.item(), m.log_prob(action)

---
### ✅ Solución — Policy de 3 capas

In [ ]:
# ✅ SOLUCIÓN — Policy de 3 capas para PixelCopter
class Policy(nn.Module):
    def __init__(self, s_size, a_size, h_size):
        super(Policy, self).__init__()
        self.fc1 = nn.Linear(s_size, h_size)          # 7 → 64
        self.fc2 = nn.Linear(h_size, h_size * 2)      # 64 → 128
        self.fc3 = nn.Linear(h_size * 2, a_size)      # 128 → 2

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return F.softmax(x, dim=1)

    def act(self, state):
        state = torch.from_numpy(state).float().unsqueeze(0).to(device)
        probs = self.forward(state).cpu()
        m = Categorical(probs)
        action = m.sample()
        return action.item(), m.log_prob(action)

### 3.3 · Hiperparámetros de PixelCopter

Comparados con CartPole, los cambios tienen una razón específica:

| Parámetro | CartPole | PixelCopter | ¿Por qué cambia? |
|-----------|----------|-------------|------------------|
| `h_size` | 16 | **64** | Entorno más complejo, necesita más capacidad |
| `n_training_episodes` | 1,000 | **50,000** | Espacio de estados más grande = más exploración |
| `gamma` | 1.0 | **0.99** | Descuento ligero — vuelos largos, el futuro importa un poco menos |
| `lr` | 1e-2 | **1e-4** | Tasa menor — red más grande, gradientes más ruidosos → pasos más pequeños |


In [ ]:
pixelcopter_hyperparameters = {
    "h_size": 64,
    "n_training_episodes": 50_000,
    "n_evaluation_episodes": 10,
    "max_t": 10_000,
    "gamma": 0.99,
    "lr": 1e-4,
    "env_id": env_id,
    "state_space": s_size,
    "action_space": a_size,
}

pixelcopter_policy    = Policy(
    pixelcopter_hyperparameters["state_space"],
    pixelcopter_hyperparameters["action_space"],
    pixelcopter_hyperparameters["h_size"]
).to(device)

pixelcopter_optimizer = optim.Adam(
    pixelcopter_policy.parameters(),
    lr=pixelcopter_hyperparameters["lr"]
)

print("✅ Policy PixelCopter creada")
print(f"Parámetros entrenables: {sum(p.numel() for p in pixelcopter_policy.parameters()):,}")

### 3.4 · Entrenar PixelCopter

> ⚠️ 50,000 episodios tarda **~60 minutos con GPU T4**.
> Puedes reducir a 10,000 para una prueba rápida — el agente aprenderá algo
> pero probablemente no alcanzará la meta de certificación (≥5).

In [ ]:
print("Entrenando PixelCopter (50,000 episodios)...")
print("Esto tardará ~60 min con GPU. Observa cómo sube la puntuación cada 1000 episodios.")
scores_pixelcopter = reinforce(
    pixelcopter_policy,
    pixelcopter_optimizer,
    pixelcopter_hyperparameters["n_training_episodes"],
    pixelcopter_hyperparameters["max_t"],
    pixelcopter_hyperparameters["gamma"],
    print_every=1000
)
print("\n✅ Entrenamiento PixelCopter completado")

### 3.5 · Evaluar PixelCopter

In [ ]:
mean_reward_pc, std_reward_pc = evaluate_agent(
    eval_env,
    pixelcopter_hyperparameters["max_t"],
    pixelcopter_hyperparameters["n_evaluation_episodes"],
    pixelcopter_policy
)
print(f"PixelCopter — Recompensa media: {mean_reward_pc:.2f} ± {std_reward_pc:.2f}")

if mean_reward_pc >= 5:
    print("✅ ¡Meta de certificación alcanzada! (≥5)")
elif mean_reward_pc >= 0:
    print("🟡 El agente vuela, pero necesita más episodios para la certificación.")
else:
    print("🔴 El agente aún no controla el vuelo.")

### 3.6 · Ver al helicóptero en video

In [ ]:
print("Grabando video de PixelCopter...")
video_pc = grabar_video_policy('Pixelcopter-PLE-v0', pixelcopter_policy, 'replay_pixelcopter.mp4', fps=30)
Video(video_pc, embed=True, width=400)

---
## 4 · Curva de aprendizaje comparativa

Graficamos la media móvil de ambos entornos en la misma escala para ver visualmente
la diferencia de complejidad entre CartPole y PixelCopter.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

# CartPole
rolling_cp = [np.mean(scores_cartpole[max(0, i-100):i+1])
              for i in range(len(scores_cartpole))]
ax1.plot(scores_cartpole, alpha=0.2, color='#3b82f6')
ax1.plot(rolling_cp, color='#22d3ee', linewidth=2)
ax1.axhline(y=350, color='#22c55e', linestyle='--', alpha=0.7, label='Meta ≥350')
ax1.set_title("CartPole-v1 (1,000 episodios)")
ax1.set_xlabel("Episodio")
ax1.set_ylabel("Puntuación")
ax1.legend()
ax1.annotate(f"Final: {np.mean(scores_cartpole[-100:]):.0f}",
             xy=(len(scores_cartpole)-1, np.mean(scores_cartpole[-100:])),
             xytext=(-80, -20), textcoords="offset points",
             color="#22d3ee", fontsize=9)

# PixelCopter
rolling_pc = [np.mean(scores_pixelcopter[max(0, i-1000):i+1])
              for i in range(len(scores_pixelcopter))]
ax2.plot(scores_pixelcopter, alpha=0.1, color='#a855f7')
ax2.plot(rolling_pc, color='#c084fc', linewidth=2)
ax2.axhline(y=5, color='#22c55e', linestyle='--', alpha=0.7, label='Meta ≥5')
ax2.set_title("PixelCopter (50,000 episodios)")
ax2.set_xlabel("Episodio")
ax2.set_ylabel("Puntuación")
ax2.legend()
ax2.annotate(f"Final: {np.mean(scores_pixelcopter[-1000:]):.1f}",
             xy=(len(scores_pixelcopter)-1, np.mean(scores_pixelcopter[-1000:])),
             xytext=(-100, -20), textcoords="offset points",
             color="#c084fc", fontsize=9)

plt.tight_layout()
plt.savefig("curvas_comparativas.png", dpi=100)
plt.show()
print("Observa: CartPole converge en ~500 episodios.")
print("PixelCopter necesita miles de episodios para mostrar mejora sostenida.")
print("Ambos usan exactamente la misma función reinforce() — la complejidad es del entorno.")

---

# 🔒 Sección opcional — Publicar en el Hub

> Requiere cuenta de HuggingFace.
> Criterios de certificación: CartPole ≥ 350 · PixelCopter ≥ 5

---

## [OPCIONAL] Publicar en el Hugging Face Hub

### Paso 1: Iniciar sesión

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

### Paso 2: Funciones de ayuda para el Hub

No necesitas entender este código ahora — es infraestructura para publicar el modelo.

In [ ]:
from huggingface_hub import HfApi, snapshot_download
from huggingface_hub.repocard import metadata_eval_result, metadata_save
from pathlib import Path
import datetime, json, tempfile

def record_video(env, policy, out_directory, fps=30):
    images = []
    done  = False
    state = env.reset()
    img   = env.render(mode='rgb_array')
    images.append(img)
    while not done:
        action, _ = policy.act(state)
        state, reward, done, info = env.step(action)
        img = env.render(mode='rgb_array')
        images.append(img)
    imageio.mimsave(out_directory, [np.array(img) for img in images], fps=fps)

def push_to_hub(repo_id, model, hyperparameters, eval_env, video_fps=30):
    _, repo_name = repo_id.split('/')
    api      = HfApi()
    repo_url = api.create_repo(repo_id=repo_id, exist_ok=True)
    with tempfile.TemporaryDirectory() as tmp:
        local = Path(tmp)
        torch.save(model, local / 'model.pt')
        with open(local / 'hyperparameters.json', 'w') as f:
            json.dump(hyperparameters, f)
        mean_r, std_r = evaluate_agent(
            eval_env, hyperparameters['max_t'],
            hyperparameters['n_evaluation_episodes'], model)
        eval_data = {'env_id': hyperparameters['env_id'],
                     'mean_reward': mean_r,
                     'eval_datetime': datetime.datetime.now().isoformat()}
        with open(local / 'results.json', 'w') as f:
            json.dump(eval_data, f)
        metadata = {'tags': [hyperparameters['env_id'], 'reinforce',
                              'reinforcement-learning', 'custom-implementation']}
        eval_meta = metadata_eval_result(
            model_pretty_name=repo_name,
            task_pretty_name='reinforcement-learning',
            task_id='reinforcement-learning',
            metrics_pretty_name='mean_reward',
            metrics_id='mean_reward',
            metrics_value=f'{mean_r:.2f} +/- {std_r:.2f}',
            dataset_pretty_name=hyperparameters['env_id'],
            dataset_id=hyperparameters['env_id'])
        metadata = {**metadata, **eval_meta}
        video_path = local / 'replay.mp4'
        record_video(eval_env, model, video_path, video_fps)
        readme = f'# Reinforce Agent — {hyperparameters["env_id"]}\n'
        readme_path = local / 'README.md'
        readme_path.write_text(readme)
        metadata_save(readme_path, metadata)
        api.upload_folder(repo_id=repo_id, folder_path=local, path_in_repo='.')
    print(f"Modelo publicado: {repo_url}")

### Paso 3: Publicar CartPole y PixelCopter

In [ ]:
username = ''   # ← pon tu usuario de HuggingFace

# Publicar CartPole
push_to_hub(
    f'{username}/Reinforce-CartPole-v1',
    cartpole_policy,
    cartpole_hyperparameters,
    gym.make('CartPole-v1')
)

# Publicar PixelCopter
push_to_hub(
    f'{username}/Reinforce-PixelCopter-PLE-v0',
    pixelcopter_policy,
    pixelcopter_hyperparameters,
    gym.make('Pixelcopter-PLE-v0')
)

---

## 🎉 ¡Notebook completado!

Implementaste REINFORCE desde cero en PyTorch — un algoritmo real de Deep RL.
Lo que construiste:

1. **`Policy`** — red neuronal que implementa π<sub>θ</sub> con softmax
2. **Encontraste y corregiste el bug de `act()`** — diferencia entre determinista y estocástica
3. **`reinforce()`** — el bucle completo: episodio → retorno hacia atrás → pérdida negativa → gradiente
4. Entrenaste y evaluaste en **CartPole** (simple) y **PixelCopter** (complejo)
5. Visualizaste cómo la **misma función** resuelve ambos con diferentes hiperparámetros

### ¿Qué sigue?

- 📊 **Módulo 5:** RL en entornos 3D con Unity ML-Agents
- 📓 **Notebook 5:** SnowballTarget y Pirámides con curiosidad intrínseca

---
*Aicraft · aicraft.mx · Jose Miguel Lara Rangel*